In [ ]:
import numpy as np
import random
from dataclasses import dataclass
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error
from sklearn.pipeline import Pipeline

In [ ]:
SEED = 67
np.random.seed(SEED) #controls randomness from numpy 
random.seed(SEED) #controls radomness from python 

In [ ]:
ACTIONS = {
    0: (-1,  0), #north  
    1: ( 0,  1), #east   
    2: ( 1,  0), #south  
    3: ( 0, -1), #west   
}
ACTION_NAMES = ["N", "E", "S", "W"] #for readable prints like confusion matrix labels
N_ACTIONS = 4

In [ ]:
@dataclass
class StepResult:
    next_state: int
    reward: float
    done: bool


class MazeEnv:
# grid : 0 = free, 1 = obstacle

    def __init__(self, grid, start_rc=None, goal_rc=None,
                 step_penalty=-1.0, wall_penalty=-5.0, goal_reward=100.0,
                 shaping=True):

        self.grid = np.array(grid, dtype=int)
        self.R, self.C = self.grid.shape

        #free cells list
        self.free_cells = [(r, c) for r in range(self.R) for c in range(self.C) if self.grid[r, c] == 0]
        self.n_states = len(self.free_cells)

        #mapping (r,c) <-> state_id
        self.rc_to_state = {rc: i for i, rc in enumerate(self.free_cells)}
        self.state_to_rc = {i: rc for i, rc in enumerate(self.free_cells)}

        #rewards
        self.step_penalty = float(step_penalty)  # small penalty each move, encouraging fewer steps
        self.wall_penalty = float(wall_penalty)  # bigger penalty for hitting wall/obstacle
        self.goal_reward = float(goal_reward)    # big reward for reaching goal

        self.shaping = bool(shaping)  #distance shaping helps learning 

        self.start_state = None
        self.goal_state = None
        self.state = None

        if start_rc is not None:
            self.set_start(start_rc)
        if goal_rc is not None:
            self.set_goal(goal_rc)

    def set_start(self, start_rc):
        start_rc = tuple(start_rc)
        if start_rc not in self.rc_to_state:
            raise ValueError(f"Start {start_rc} is not a free cell.")
        self.start_state = self.rc_to_state[start_rc]

    def set_goal(self, goal_rc):
        goal_rc = tuple(goal_rc)
        if goal_rc not in self.rc_to_state:
            raise ValueError(f"Goal {goal_rc} is not a free cell.")
        self.goal_state = self.rc_to_state[goal_rc]

    def reset(self, start_rc=None, goal_rc=None):
        if start_rc is not None:
            self.set_start(start_rc)
        if goal_rc is not None:
            self.set_goal(goal_rc)
        if self.start_state is None or self.goal_state is None:
            raise ValueError("Need start and goal set before reset().")

        self.state = self.start_state
        return self.state

    def manhattan(self, s):
        # Manhattan distance to goal (used for shaping)
        r, c = self.state_to_rc[s]
        gr, gc = self.state_to_rc[self.goal_state]
        return abs(r - gr) + abs(c - gc)

    def step(self, action_id):
        r, c = self.state_to_rc[self.state]
        dr, dc = ACTIONS[action_id]
        nr, nc = r + dr, c + dc

        #if out of bounds or obstacle, stay and get penalty
        if not (0 <= nr < self.R and 0 <= nc < self.C) or self.grid[nr, nc] == 1:
            return StepResult(next_state=self.state, reward=self.wall_penalty, done=False)

        ns = self.rc_to_state[(nr, nc)]
        done = (ns == self.goal_state)

        reward = self.step_penalty
        if done:
            reward += self.goal_reward

        #shaping : reward moving closer, penalize moving away a bit
        if self.shaping and not done:
            reward += 0.5 * (self.manhattan(self.state) - self.manhattan(ns))

        self.state = ns
        return StepResult(next_state=ns, reward=float(reward), done=done)

    def plot_path(self, path_states, title="Path", save_path=None):
        #maze image : obstacles black, free white, start green, goal red
        img = np.ones((self.R, self.C, 3), dtype=float)
        img[self.grid == 1] = [0, 0, 0]  #obstacles = black

        sr, sc = self.state_to_rc[self.start_state]
        gr, gc = self.state_to_rc[self.goal_state]
        img[sr, sc] = [0.2, 1.0, 0.2]  #start = green
        plt.scatter([gc], [gr], marker="*", s=300, c="red")

        plt.figure(figsize=(6, 6))
        plt.imshow(img, interpolation="nearest")
        plt.title(title)
        plt.xticks(range(self.C))
        plt.yticks(range(self.R))
        plt.grid(True)

        if path_states:
            coords = [self.state_to_rc[s] for s in path_states]
            ys = [r for r, c in coords]
            xs = [c for r, c in coords]
            plt.plot(xs, ys, marker="o", linewidth=2)

        if save_path is not None:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()

In [ ]:
class QLearner:
    def __init__(self, n_states,
                 alpha=0.15, gamma=0.95,
                 eps_start=1.0, eps_min=0.05, eps_decay=0.999,
                 seed=SEED):

        self.Q = np.zeros((n_states, N_ACTIONS), dtype=float)
        self.alpha = float(alpha)
        self.gamma = float(gamma)

        self.eps = float(eps_start)
        self.eps_min = float(eps_min)
        self.eps_decay = float(eps_decay)

        self.rng = np.random.RandomState(seed)  #local rng = cleaner repeatability

    def choose_action(self, s):
        #epsilon-greedy exploration
        if self.rng.rand() < self.eps:
            return int(self.rng.randint(N_ACTIONS))
        q = self.Q[s]
        best = np.flatnonzero(q == np.max(q))
        return int(self.rng.choice(best))

    def update(self, s, a, r, ns, done):
        # Bellman update
        target = r if done else (r + self.gamma * np.max(self.Q[ns]))
        self.Q[s, a] = (1 - self.alpha) * self.Q[s, a] + self.alpha * target

    def decay_eps(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)


def train_qlearning(env: MazeEnv,
                    episodes=4000, max_steps=400,
                    alpha=0.15, gamma=0.95,
                    eps_start=1.0, eps_min=0.05, eps_decay=0.999,
                    seed=SEED):

    learner = QLearner(env.n_states, alpha=alpha, gamma=gamma,
                       eps_start=eps_start, eps_min=eps_min, eps_decay=eps_decay,
                       seed=seed)

    rewards_hist = []
    success_hist = []

    all_states = np.arange(env.n_states)

    for _ in range(episodes):
        #random start but not the goal
        s0 = int(np.random.choice(all_states))
        while s0 == env.goal_state:
            s0 = int(np.random.choice(all_states))

        env.start_state = s0
        s = env.reset()

        total_r = 0.0
        success = 0

        for _ in range(max_steps):
            a = learner.choose_action(s)
            res = env.step(a)
            learner.update(s, a, res.reward, res.next_state, res.done)
            total_r += res.reward
            s = res.next_state
            if res.done:
                success = 1
                break

        learner.decay_eps()
        rewards_hist.append(total_r)
        success_hist.append(success)

    return learner, np.array(rewards_hist), np.array(success_hist)


def greedy_rollout(env: MazeEnv, Q, start_rc, goal_rc, max_steps=500):
    #follow greedy policy using learned Q
    env.reset(start_rc=start_rc, goal_rc=goal_rc)
    path = [env.state]

    for _ in range(max_steps):
        q = Q[env.state]
        best = np.flatnonzero(q == np.max(q))
        a = int(np.random.choice(best))
        res = env.step(a)
        path.append(env.state)
        if res.done:
            return path, True
    return path, False

In [ ]:
def build_dataset_for_all_configs(grid, episodes_per_goal=4000):
    env = MazeEnv(grid, shaping=True)

    free_cells = env.n_states
    print("Free cells =", free_cells)
    if free_cells != 28:
        print("WARNING : Free cells is not 28. That usually means the maze grid is wrong.")
        print("Fix MAZE_GRID until Free cells = 28 so the dataset becomes 756 rows.")

    Q_by_goal = {}
    reward_curves = {}

    #train 1 Q-table per goal -> efficient & clean
    for goal_rc in env.free_cells:
        env.set_goal(goal_rc)
        learner, rewards_hist, _ = train_qlearning(env, episodes=episodes_per_goal)
        Q_by_goal[goal_rc] = learner.Q
        reward_curves[goal_rc] = rewards_hist

    # dataset : (cur_x, cur_y, goal_x, goal_y) -> best_action
    X = []
    y = []

    for goal_rc in env.free_cells:
        Q = Q_by_goal[goal_rc]
        for cur_rc in env.free_cells:
            if cur_rc == goal_rc:
                continue
            s = env.rc_to_state[cur_rc]
            a = int(np.argmax(Q[s]))
            cur_r, cur_c = cur_rc
            goal_r, goal_c = goal_rc
            X.append([cur_c, cur_r, goal_c, goal_r])  # x,y,gx,gy
            y.append(a)

    X = np.array(X, dtype=float)
    y = np.array(y, dtype=int)

    print("Dataset rows =", len(X))
    expected = free_cells * (free_cells - 1)
    print("Expected rows =", expected)

    if free_cells == 28 and len(X) != 756:
        raise RuntimeError("Expected dataset size 756 but got something else.")

    return env, Q_by_goal, reward_curves, X, y


def save_dataset_csv(X, y, out_csv):
    header = "cur_x,cur_y,goal_x,goal_y,opt_action"
    data = np.column_stack([X, y])
    np.savetxt(out_csv, data, delimiter=",", header=header, comments="", fmt="%.0f")
    print("Saved dataset:", out_csv)

In [ ]:
def train_mlp_controller(X, y, test_size=0.25):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=SEED,
        stratify=y
    )

    model = Pipeline([
        ("scaler", StandardScaler()),  #helps MLP converge better
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(64, 64),
            activation="relu",
            solver="adam",
            learning_rate_init=0.001,
            max_iter=2000,
            random_state=SEED
        ))
    ])

    model.fit(X_train, y_train)

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    acc_train = accuracy_score(y_train, pred_train)
    acc_test = accuracy_score(y_test, pred_test)
    rmse_test = mean_squared_error(y_test, pred_test) ** 0.5  #RMSE on labels (0..3)

    print("Train accuracy:", acc_train)
    print("Test  accuracy:", acc_test)
    print("Test  RMSE:", rmse_test)
    print("\nConfusion matrix (test):\n", confusion_matrix(y_test, pred_test))
    print("\nClassification report (test):\n", classification_report(y_test, pred_test, target_names=ACTION_NAMES))

    metrics = {
        "acc_train": float(acc_train),
        "acc_test": float(acc_test),
        "rmse_test": float(rmse_test)
    }
    return model, metrics


def nn_choose_action(model, cur_rc, goal_rc):
    cur_r, cur_c = cur_rc
    goal_r, goal_c = goal_rc
    x = np.array([[cur_c, cur_r, goal_c, goal_r]], dtype=float)
    return int(model.predict(x)[0])


def rollout_nn(env: MazeEnv, model, start_rc, goal_rc, max_steps=500):
    env.reset(start_rc=start_rc, goal_rc=goal_rc)
    path = [env.state]

    for _ in range(max_steps):
        cur_rc = env.state_to_rc[env.state]
        a = nn_choose_action(model, cur_rc, goal_rc)
        res = env.step(a)
        path.append(env.state)
        if res.done:
            return path, True
    return path, False

In [ ]:
def read_maze_file(filepath):
    def clean(line):
        return line.split("#", 1)[0].strip()  #remove anything after '#'

    with open(filepath, "r") as f:
        raw = [clean(ln) for ln in f.readlines()]
        lines = [ln for ln in raw if ln]  #drop empty lines

    R, C = map(int, lines[0].split())
    sr, sc = map(int, lines[1].split())
    gr, gc = map(int, lines[2].split())

    grid = []
    for i in range(R):
        row = list(map(int, lines[3 + i].split()))
        if len(row) != C:
            raise ValueError("Maze row length mismatch in file.")
        grid.append(row)

    start_rc = (sr - 1, sc - 1)
    goal_rc = (gr - 1, gc - 1)
    return grid, start_rc, goal_rc


def solve_maze_from_file(filepath, episodes=5000, max_steps=600, out_dir="results_task3"):
    grid, start_rc, goal_rc = read_maze_file(filepath)
    env = MazeEnv(grid, start_rc=start_rc, goal_rc=goal_rc, shaping=True)

    learner, rewards_hist, success_hist = train_qlearning(
        env,
        episodes=episodes,
        max_steps=max_steps
    )

    path, ok = greedy_rollout(env, learner.Q, start_rc, goal_rc, max_steps=max_steps)

    #output paths
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    base = Path(filepath).stem

    #save reward curve
    plt.figure()
    plt.plot(rewards_hist)
    plt.title(f"Q-learning reward curve ({base})")
    plt.xlabel("Episode")
    plt.ylabel("Total reward")
    reward_png = out_dir / f"{base}_reward_curve.png"
    plt.savefig(reward_png, dpi=300, bbox_inches="tight")
    plt.show()

    #save path plot
    path_png = out_dir / f"{base}_path.png"
    env.plot_path(path, title=f"Greedy path ({base})", save_path=str(path_png))

    return {
        "file": str(filepath),
        "ok": bool(ok),
        "steps": int(len(path) - 1),
        "reward_png": str(reward_png),
        "path_png": str(path_png)
    }


def save_text(filename, lines):
    with open(filename, "w") as f:
        for ln in lines:
            f.write(str(ln) + "\n")
    print("Saved:", filename)

In [ ]:
MAZE_GRID = [
    [0, 0, 1, 0, 1, 0],  
    [0, 0, 0, 0, 1, 0], 
    [0, 1, 1, 0, 0, 0],  
    [0, 0, 0, 1, 0, 0],  
    [0, 0, 0, 0, 1, 0],  
    [0, 0, 1, 0, 0, 0],  
]

In [ ]:
#save everyhting
out12 = Path("results_task1_task2")
out12.mkdir(parents=True, exist_ok=True)

#build dataset 
env12, Q_by_goal, reward_curves, X, y = build_dataset_for_all_configs(MAZE_GRID, episodes_per_goal=4000)

#save dataset 
save_dataset_csv(X, y, out_csv=str(out12 / "qlearning_dataset_756.csv"))

# pick demo start/goal 
start_rc = env12.free_cells[0]
goal_rc = env12.free_cells[-1]
Q = Q_by_goal[goal_rc]

path_q, ok_q = greedy_rollout(env12, Q, start_rc, goal_rc, max_steps=400)
print("\nTask 1 demo (Q-learning): reached goal?", ok_q, "| steps:", len(path_q) - 1)

env12.plot_path(path_q,
                title="Task 1: Q-learning greedy path (demo)",
                save_path=str(out12 / "task1_qlearning_path.png"))

#save an example goal 
plt.figure()
plt.plot(reward_curves[goal_rc])
plt.title("Task 1: Reward curve (example goal)")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.savefig(out12 / "task1_reward_curve_example.png", dpi=300, bbox_inches="tight")
plt.show()

#training (NN)
nn_model, nn_metrics = train_mlp_controller(X, y, test_size=0.25)

#demo path (NN) 
path_nn, ok_nn = rollout_nn(env12, nn_model, start_rc, goal_rc, max_steps=400)
print("\nTask 2 demo (NN): reached goal?", ok_nn, "| steps:", len(path_nn) - 1)

env12.plot_path(path_nn,
                title="Task 2: Neural Network path (demo)",
                save_path=str(out12 / "task2_nn_path.png"))

#freeze numbers 
save_text(out12 / "summary_results.txt", [
    f"SEED = {SEED}",
    f"Free cells = {env12.n_states}",
    f"Dataset rows = {len(X)}",
    f"Task1_demo_reached_goal = {ok_q}, steps = {len(path_q) - 1}",
    f"Task2_demo_reached_goal = {ok_nn}, steps = {len(path_nn) - 1}",
    f"NN acc_train = {nn_metrics['acc_train']}",
    f"NN acc_test  = {nn_metrics['acc_test']}",
    f"NN rmse_test = {nn_metrics['rmse_test']}",
])

print("\n✅ Task 1+2 outputs saved in:", out12.resolve())

In [ ]:
INPUT_DIR = Path(".") 
maze1 = INPUT_DIR / "maze1.txt"
maze2 = INPUT_DIR / "maze2.txt"

print("\n--- Task 3 auto run ---")
print("Looking for :", maze1)
print("Looking for :", maze2)

task3_results = []
if maze1.exists():
    task3_results.append(solve_maze_from_file(str(maze1), out_dir="results_task3"))
else:
    print("maze1.txt not found.")

if maze2.exists():
    task3_results.append(solve_maze_from_file(str(maze2), out_dir="results_task3"))
else:
    print("maze2.txt not found.")

# save task3 results summary
out3 = Path("results_task3")
out3.mkdir(exist_ok=True)
save_text(out3 / "task3_summary.txt", task3_results)

print("\nTask 3 outputs saved in :", out3.resolve())